In [1]:
import pandas as pd
import cv2
import glob
import math

%pylab inline

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [2]:
import os, sys, codecs, glob
from PIL import Image, ImageDraw

import numpy as np
import pandas as pd
import cv2

import torch
torch.backends.cudnn.benchmark = False
# torch.backends.cudnn.enabled = False

import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
from torch.utils.data.dataset import Dataset

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [3]:
class XunFeiDataset(Dataset):
    def __init__(self, img_path, img_group, transform):
        self.img_path = img_path
        self.transform = transform
        self.group = img_group

    def __getitem__(self, index):
        img = Image.open(self.img_path[index]).convert('RGB')
        
        if self.transform is not None:
            img = self.transform(img)
        
        return img, self.group[index]

    def __len__(self):
        return len(self.img_path)

In [10]:
train_imgs = glob.glob("./大熊猫个体识别数据集（公开）/数据集构建脚本/iPanda50/cropped_parts/*/*/*.jpg")
train_labels = [x.split('/')[-2] for x in train_imgs]
lbl = LabelEncoder()
encoded_labels = lbl.fit_transform(train_labels)

# 划分训练集和验证集（80%训练，20%验证）
train_paths, val_paths, train_labels_encoded, val_labels_encoded = train_test_split(
    train_imgs, 
    encoded_labels, 
    test_size=0.1,          # 验证集比例
    random_state=42,        # 随机种子，保证结果可复现
    # stratify=encoded_labels # 分层采样，保持类别分布一致
)

In [12]:
len(set(encoded_labels))

50

In [13]:
train_loader = torch.utils.data.DataLoader(
    XunFeiDataset(train_paths, train_labels_encoded,
                        transforms.Compose([
                        transforms.Resize((300, 300)),
                        transforms.RandomHorizontalFlip(),
                        transforms.RandomVerticalFlip(),
                        transforms.ToTensor(),
                        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    ),
    batch_size=30, shuffle=True, num_workers=5,
)

val_loader = torch.utils.data.DataLoader(
    XunFeiDataset(val_paths, val_labels_encoded,
                        transforms.Compose([
                        transforms.Resize((300, 300)),
                        transforms.ToTensor(),
                        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    ),
    batch_size=30, shuffle=False, num_workers=5,
)

In [14]:
import timm
timm.create_model('efficientnet_b0', num_classes=4067, pretrained=True).classifier

Linear(in_features=1280, out_features=4067, bias=True)

In [15]:
import timm

class XunFeiNet(nn.Module):
    def __init__(self):
        super(XunFeiNet, self).__init__()
                
        model = timm.create_model('efficientnet_b0', num_classes=50, 
                          pretrained=True)
        # model.classifier = torch.nn.Identity()
        self.model = model
        
    def forward(self, img, labels=None):        
        feat = self.model(img)
        
        # feat = F.normalize(feat)
        # if labels is not None:
        #     return self.margin(feat, labels)
        return feat
    
model = XunFeiNet().cuda()
model

XunFeiNet(
  (model): EfficientNet(
    (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
          (conv_pw): Conv2d(32, 16, kernel_size=(1, 1)

In [16]:
def train(train_loader, model, criterion, optimizer, epoch):
    model.train()

    for i, (input, target) in enumerate(train_loader):
        input = input.cuda(non_blocking=True)
        target = target.cuda(non_blocking=True)

        output = model(input, target)
        loss = criterion(output, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if i % 40 == 0:
            print(loss.item())
            
def validate(val_loader, model):
    model.eval()
    
    val_feats = []
    with torch.no_grad():
        end = time.time()
        for i, (input, target) in enumerate(val_loader):
            input = input.cuda()
            target = target.cuda()

            # compute output
            output = model(input)
            val_feats.append(output.data.cpu().numpy())
    return val_feats

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
import numpy as np

criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.Adam(model.parameters(), 0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.85)
best_acc = 0.0

def validate_classification(val_loader, model):
    """
    验证函数：直接返回预测结果和真实标签
    """
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.cuda()
            labels = labels.cuda()
            
            # 前向传播
            outputs = model(images)
            # 获取预测类别
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return all_preds, all_labels

for epoch in range(20):
    print('Epoch: ', epoch)

    # 训练
    train(train_loader, model, criterion, optimizer, epoch)
    scheduler.step()
    
    # 验证
    val_preds, val_labels = validate_classification(val_loader, model)
    
    # 计算准确率
    acc = accuracy_score(val_labels, val_preds)
    
    print(f'Validation Accuracy: {acc:.4f}')
    
    # 保存最佳模型
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'New best model saved with accuracy: {best_acc:.4f}')

print(f'Best Validation Accuracy: {best_acc:.4f}')

Epoch:  0
4.286554336547852
3.44777512550354
3.1265721321105957
1.8922054767608643
2.521996259689331
Validation Accuracy: 0.3455
New best model saved with accuracy: 0.3455
Epoch:  1
1.8650166988372803
1.9165946245193481
2.1329736709594727
1.8014529943466187
1.4494798183441162
Validation Accuracy: 0.4182
New best model saved with accuracy: 0.4182
Epoch:  2
1.0002905130386353
1.3216630220413208
1.4521677494049072
1.6011970043182373
1.2611764669418335
Validation Accuracy: 0.4673
New best model saved with accuracy: 0.4673
Epoch:  3
0.9654054641723633
0.8419920206069946
0.6806841492652893
1.0140137672424316
1.4089730978012085
Validation Accuracy: 0.5200
New best model saved with accuracy: 0.5200
Epoch:  4
0.6329771280288696
0.2916116714477539
0.6686012148857117
0.5922451615333557
0.4284713864326477
Validation Accuracy: 0.5091
Epoch:  5
0.17814341187477112
0.2351454347372055
0.38965368270874023
0.5993814468383789
0.4549658000469208
Validation Accuracy: 0.5164
Epoch:  6
0.2194770723581314
0.1

KeyboardInterrupt: 

In [18]:
test_path = glob.glob('./大熊猫个体识别数据集（公开）/测试集/*/*.jpg')
test_path.sort()
test_path = np.array(test_path)

test_loader = torch.utils.data.DataLoader(
    XunFeiDataset(test_path, [0]*len(test_path),
                        transforms.Compose([
                        transforms.Resize((300, 300)),
                        transforms.ToTensor(),
                        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    ),
    batch_size=50, shuffle=False, num_workers=5,
)

In [19]:
model.eval()
test_feats = []
with torch.no_grad():
    for data in test_loader:
        data = data[0].cuda()
        feat = model(data)
        test_feats.append(feat.data.cpu().numpy())
        
test_feats = np.vstack(test_feats)

In [ ]:
test_submit = pd.DataFrame([x.split("/")[-1] for x in test_path], columns=['image_id'])
test_submit['predicted_id'] = lbl.inverse_transform(test_feats.argmax(1))
test_submit
test_submit.to_csv('submit_qe.csv',index=None)